### Welcome to Week 5 Day 1

AutoGen AgentChat!

This should look simple and familiar, because it has a lot in common with Crew and OpenAI Agents SDK

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### First concept: the Model

In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [3]:
from autogen_ext.models.ollama import OllamaChatCompletionClient
ollamamodel_client = OllamaChatCompletionClient(model="llama3.2")

### Second concept: The Message

In [16]:
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="I'd like to go to Rome", source="user")
message

TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 20, 20, 36, 5, 920393, tzinfo=datetime.timezone.utc), content="I'd like to go to Rome", type='TextMessage')

### Third concept: The Agent

In [5]:
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(
    name="airline_agent",
    model_client=model_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers.",
    model_client_stream=True
)

### Put it all together with on_messages

In [6]:
from autogen_core import CancellationToken

# Send the message to the agent and get a response
# on_messages() is the main method to interact with AutoGen agents
# It takes a list of messages and returns a response object
response = await agent.on_messages([message], cancellation_token=CancellationToken())

# Extract the actual text content from the response
# response.chat_message contains the final message from the agent
response.chat_message.content

'Great choice! Rome: where the pasta is a religious experience and the traffic is a modern-day gladiator arena. Buckle up! 🍝🏛️'

### Let's make a local database of ticket prices

In [7]:
import os
import sqlite3

# Delete existing database file if it exists
if os.path.exists("tickets.db"):
    os.remove("tickets.db")

# Create the database and the table
conn = sqlite3.connect("tickets.db")
c = conn.cursor()
c.execute("CREATE TABLE cities (city_name TEXT PRIMARY KEY, round_trip_price REAL)")
conn.commit()
conn.close()

In [8]:
# Populate our database
def save_city_price(city_name, round_trip_price):
    """Save or update a city's round-trip ticket price in the database.
    
    Uses REPLACE INTO to either insert new cities or update existing ones.
    City names are stored in lowercase for case-insensitive lookups.
    """
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("REPLACE INTO cities (city_name, round_trip_price) VALUES (?, ?)", (city_name.lower(), round_trip_price))
    conn.commit()
    conn.close()

# Populate the database with sample European city ticket prices
# These represent fictional round-trip prices from a default airport
save_city_price("London", 299)
save_city_price("Paris", 399)
save_city_price("Rome", 499)
save_city_price("Madrid", 550)
save_city_price("Barcelona", 580)
save_city_price("Berlin", 525)

In [15]:
# Method to get price for a city
# This is what we consider as a tool and might have other decorators and learning curve in other frameworks but this is lightweight
def get_city_price(city_name: str) -> float | None:
    """ Get the roundtrip ticket price to travel to the city """
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city_name.lower(),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None

In [10]:
get_city_price("Rome")

499.0

In [11]:
from autogen_agentchat.agents import AssistantAgent

smart_agent = AssistantAgent(
    name="smart_airline_agent",
    model_client=ollamamodel_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers, including the price of a roundtrip ticket.",
    model_client_stream=True,
    tools=[get_city_price],
    reflect_on_tool_use=True
)

In [17]:
# Send the user's message to the smart agent, which has access to the get_city_price tool
# The agent will analyze the message content ("I'd like to go to London") and call the tool with "London"
# The city name is extracted from the user's message, not explicitly passed in the system message
print(f"Sending message: {message.content}")  # Show the message content for clarity
response = await smart_agent.on_messages([message], cancellation_token=CancellationToken())

# Print intermediate messages (like tool calls) for debugging/visibility
for inner_message in response.inner_messages:
    print(inner_message.content)

# Get the final response from the agent, which should include the price
response.chat_message.content

Sending message: I'd like to go to Rome
[FunctionCall(id='0', arguments='{"city_name": "Rome"}', name='get_city_price')]
[FunctionExecutionResult(content='499.0', name='get_city_price', call_id='0', is_error=False)]
[FunctionCall(id='0', arguments='{"city_name": "Rome"}', name='get_city_price')]
[FunctionExecutionResult(content='499.0', name='get_city_price', call_id='0', is_error=False)]


"You're a repeat offender, aren't you? Alright, alright! You want to fly to Rome again... and again... Okay, fine! We'll give you the same deal:\n\nRoundtrip ticket price: $499 (but if you book multiple flights, we might start charging extra for your love of Italy)"